In [ ]:
import polars as pl
import matplotlib.pyplot as plt
import numpy as np

# KEY EDA to do

Project will involeve the flows too and from each user and we will look at binning the flows.

1) Basic checks
    Checks for duplicates number of source and destination computers, time window etc.
2) Frequency checks
    Checks for the distributions of user sent and recieved signal activity, frequency by human and machine.
3) Do basic statistical tests for periodicity across users and in human and machine users. (randomly sample if necessary)
4) investigate the effects of binning

list 2
1) Invesitgate the flows too and from each user how many go from each machine in a given time period.
2) Investigate variance of a machines flows over time.
3) Investigate the effect of binning width has on machine flow bias and variance.
4) Aim to indentify both human and machine operated accounts.
5) Static clustering

#### Ideas
- Idea smooth the distribution of mus towards the other distribution. I.e. for each user we will have a value of mu for each bin. Then for each cluster we will have an average distribution of mus. We will smooth the lower tail of the mu for a user towards the average lower tail obsered in that cluster. This way we preserve the periodic structure for each user whilst smoothing extremes.
- Pick binning based on frequency periodicity analysis.

#### Questions
- How to bin
- How to handle multiple requests within the same time period
- How to handle low frequency users
- How to smooth
- How good is the human machine classification system I have seen human users exhibiting behaviour with high requests and periodicity that are often seen in machines is this just a normal upper tail human or a machine
- Should I be dealing with users or computers
- What are the anonymous logons in the dataset

In [3]:
df = pl.scan_parquet("/home/ma/a/alb25/Project/thesis_code/data/raw/*")

In [4]:
df.head().collect()

time,source_user@domain,destination_user@domain,source_computer,destination_computer,authentication_type,logon_type,authentication_orientation,success/failure
str,str,str,str,str,str,str,str,str
"""1""","""ANONYMOUS LOGON@C586""","""ANONYMOUS LOGON@C586""","""C1250""","""C586""","""NTLM""","""Network""","""LogOn""","""Success"""
"""1""","""ANONYMOUS LOGON@C586""","""ANONYMOUS LOGON@C586""","""C586""","""C586""","""?""","""Network""","""LogOff""","""Success"""
"""1""","""C101$@DOM1""","""C101$@DOM1""","""C988""","""C988""","""?""","""Network""","""LogOff""","""Success"""
"""1""","""C1020$@DOM1""","""SYSTEM@C1020""","""C1020""","""C1020""","""Negotiate""","""Service""","""LogOn""","""Success"""
"""1""","""C1021$@DOM1""","""C1021$@DOM1""","""C1021""","""C625""","""Kerberos""","""Network""","""LogOn""","""Success"""


## Basic Checks

In [3]:
destinations = df.select('destination_computer').unique().collect()['destination_computer'].to_list()

In [4]:
sources = df.select('source_computer').unique().collect()['source_computer'].to_list()

In [ ]:
len(destinations)

15620

In [7]:
len(sources)

16006

In [ ]:
df.head().collect()

time,source_user@domain,destination_user@domain,source_computer,destination_computer,authentication_type,logon_type,authentication_orientation,success/failure
str,str,str,str,str,str,str,str,str
"""1""","""ANONYMOUS LOGON@C586""","""ANONYMOUS LOGON@C586""","""C1250""","""C586""","""NTLM""","""Network""","""LogOn""","""Success"""
"""1""","""ANONYMOUS LOGON@C586""","""ANONYMOUS LOGON@C586""","""C586""","""C586""","""?""","""Network""","""LogOff""","""Success"""
"""1""","""C101$@DOM1""","""C101$@DOM1""","""C988""","""C988""","""?""","""Network""","""LogOff""","""Success"""
"""1""","""C1020$@DOM1""","""SYSTEM@C1020""","""C1020""","""C1020""","""Negotiate""","""Service""","""LogOn""","""Success"""
"""1""","""C1021$@DOM1""","""C1021$@DOM1""","""C1021""","""C625""","""Kerberos""","""Network""","""LogOn""","""Success"""


In [53]:
df.select(
    pl.col('time').cast(pl.Int64).max().alias('max_time'),
    pl.col('time').cast(pl.Int64).min().alias('min_time'),
    ).collect()

max_time,min_time
i64,i64
5011199,1


In [ ]:
# We have 58 days of data
5011199 / (3600*24)

57.99998842592593

In [6]:
cols = df.collect_schema().names()
df = df.with_columns(pl.struct(cols).hash(seed=0).alias('hash'))
df2 = df.select('hash').group_by('hash').len().top_k(by='len', k=5).head().collect()

In [ ]:
# No duplicate rows found in the data
df2

hash,len
u64,u32
10489887825803446894,1
10043600765358434665,1
17312281700719399321,1
7365033346672452035,1
6629053896398302689,1


## Human vs machine identification

In [6]:
df.head().collect()

time,source_user@domain,destination_user@domain,source_computer,destination_computer,authentication_type,logon_type,authentication_orientation,success/failure
str,str,str,str,str,str,str,str,str
"""1""","""ANONYMOUS LOGON@C586""","""ANONYMOUS LOGON@C586""","""C1250""","""C586""","""NTLM""","""Network""","""LogOn""","""Success"""
"""1""","""ANONYMOUS LOGON@C586""","""ANONYMOUS LOGON@C586""","""C586""","""C586""","""?""","""Network""","""LogOff""","""Success"""
"""1""","""C101$@DOM1""","""C101$@DOM1""","""C988""","""C988""","""?""","""Network""","""LogOff""","""Success"""
"""1""","""C1020$@DOM1""","""SYSTEM@C1020""","""C1020""","""C1020""","""Negotiate""","""Service""","""LogOn""","""Success"""
"""1""","""C1021$@DOM1""","""C1021$@DOM1""","""C1021""","""C625""","""Kerberos""","""Network""","""LogOn""","""Success"""


In [8]:
source_users = df.select('source_user@domain').unique().collect()

In [10]:
source_users = source_users.to_series().to_list()

In [11]:
source_users

['C10010$@?',
 'C20627$@DOM1',
 'U1139@DOM1',
 'SYSTEM@C20114',
 'U2513@DOM9',
 'C26818$@?',
 'SYSTEM@C11866',
 'SYSTEM@C473',
 'ANONYMOUS LOGON@C20527',
 'U6706@?',
 'LOCAL SERVICE@C9976',
 'SYSTEM@C2363',
 'U10521@?',
 'ANONYMOUS LOGON@C10574',
 'C328$@DOM1',
 'U1389@?',
 'NETWORK SERVICE@C4593',
 'U5074@DOM9',
 'U5921@?',
 'NETWORK SERVICE@C10593',
 'U8061@DOM1',
 'U6347@DOM1',
 'U9410@DOM1',
 'U10613@DOM9',
 'C21313$@DOM1',
 'U6079@?',
 'ANONYMOUS LOGON@C1685',
 'C6236$@DOM1',
 'SYSTEM@C314',
 'C1272$@DOM1',
 'C18759$@?',
 'SYSTEM@C13455',
 'C1243$@DOM1',
 'ANONYMOUS LOGON@C5412',
 'LOCAL SERVICE@C4271',
 'ANONYMOUS LOGON@C15394',
 'LOCAL SERVICE@C2800',
 'C6151$@DOM1',
 'SYSTEM@C19094',
 'C450$@DOM1',
 'C7873$@DOM1',
 'U5167@DOM1',
 'NETWORK SERVICE@C18048',
 'C19280$@DOM1',
 'U6618@DOM9',
 'U1956@?',
 'U11168@DOM3',
 'C24122$@DOM1',
 'U6333@DOM9',
 'U12043@C24564',
 'NETWORK SERVICE@C7252',
 'ANONYMOUS LOGON@C17691',
 'U7505@DOM21',
 'SYSTEM@C12257',
 'SYSTEM@C20295',
 'U7363@C48

In [12]:
# Adding user types
df = df.with_columns(pl.when(pl.col('source_user@domain').str.contains(r'^U\d+@')).then(pl.lit('human'))
          .when(pl.col('source_user@domain').str.contains(r'^C\d+\$@')).then(pl.lit('machine'))
          .when(pl.col('source_user@domain').str.contains(r'^(SYSTEM|LOCAL SERVICE|NETWORK SERVICE)@')).then(pl.lit('system'))
          .when(pl.col('source_user@domain').str.contains(r'^ANONYMOUS LOGON@')).then(pl.lit('anon'))
          .otherwise(pl.lit('other'))
          .alias('source_user_type'))

df = df.with_columns(pl.when(pl.col('destination_user@domain').str.contains(r'^U\d+@')).then(pl.lit('human'))
          .when(pl.col('destination_user@domain').str.contains(r'^C\d+\$@')).then(pl.lit('machine'))
          .when(pl.col('destination_user@domain').str.contains(r'^(SYSTEM|LOCAL SERVICE|NETWORK SERVICE)@')).then(pl.lit('system'))
          .when(pl.col('destination_user@domain').str.contains(r'^ANONYMOUS LOGON@')).then(pl.lit('anon'))
          .otherwise(pl.lit('other'))
          .alias('destination_user_type'))

In [ ]:
# Have managed to sucessfully capture all users with regexp
df.filter((pl.col('source_user@domain') == 'other') | (pl.col('destination_user@domain') == 'other')).collect()

time,source_user@domain,destination_user@domain,source_computer,destination_computer,authentication_type,logon_type,authentication_orientation,success/failure,source_user_type,destination_user_type
str,str,str,str,str,str,str,str,str,str,str


### Noteworthy findings

#### Section 1
No duplicates found


Clear distinction between human and machine users in the dataframe dermakated by $
Clear periodic behaviour observed for some machine users
Multiple requests made within identical time periods common

#### Example of clearly regular machine behaviour

![Output plot](savedplots/machine_plot1.png)
![Output plot](savedplots/machine_plot2.png)

### Typical examples of human behaviour
Less regularity is typically observed

![Output plot](savedplots/human_user1.png)
![Output plot](savedplots/human_user2.png)

### A note on domains, machines and system processes

Domains are basically big databases of usernames and passwords and permissions on a windows server. You authenticate against the domain and then are granted permissions to do stuff by the domain controler.

User activity is usually run with a user account. 

Computer accounts also exist and can be used by appications for running processes automated and not.

Windows also provides some built in accounts which can be harnessed by applications. These are the system accounts the domain here is the machine the system account is running on.

Anonymous logons are sessions where Windows did not associate the connection with a named authenticated identity. The underlying actor could be a user process, machine process, system process, or misconfigured service.

#### A note on source and destination users and machines

- source user = who you are coming from
- destination user = who you are logging/logging off in as
- source computer = where the login/logoff request comes from
- destination computer = where you are logging in /logging off too, e.g. accessing a file on a server requires you to authenticate onto a different machine

### Cleaning + Pipeline to do

- Filter the anon users as these can correspond to many different types of behaviour
- Maybe filter the system users out too
- Operate on a user not a computer level as multiple different users can share the same machine making messy behaviour on a machine level with mixture between human and machine time series observed.
- For each user, we estimate a mean and variance for every bin. These estimates are then smoothed towards cluster-average values. We first sort each user’s bin-level parameters to form a user-specific distribution. We then build a cluster-level reference distribution by averaging parameters at the same sorted position across users in the cluster. Each user’s parameter is then smoothed towards the corresponding value in this cluster distribution.
